# Web Scraping for Data Analysis
### Objective
This notebook demonstrates how to extract data from a website, clean it, and perform basic Exploratory Data Analysis (EDA).

**Project Workflow:**
1. Request HTML data from `books.toscrape.com`.
2. Parse the HTML using BeautifulSoup.
3. Loop through multiple pages (Pagination).
4. Clean and convert data types with Pandas.
5. Visualize the price distribution.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import time

# Setup for visualization
%matplotlib inline
plt.style.use('ggplot')

### Step 1: Multi-Page Scraper
We use a `for` loop to iterate through the first 5 pages and a `try-except` block to handle potential connection issues.

In [ ]:
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
all_books = []

for page in range(1, 6):
    url = base_url.format(page)
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Find all book containers
        books = soup.find_all("article", class_="product_pod")
        
        for book in books:
            title = book.h3.a["title"]
            price = book.find("p", class_="price_color").text
            availability = book.find("p", class_="instock availability").text.strip()
            
            all_books.append({
                "Title": title,
                "Price": price,
                "Status": availability
            })
        
        print(f"Page {page} scraped successfully.")
        time.sleep(1) # Polite delay
        
    except Exception as e:
        print(f"Failed to scrape page {page}: {e}")

df = pd.DataFrame(all_books)
print(f"Total books collected: {len(df)}")

### Step 2: Data Cleaning
Scraped data is always 'string' format. We need to remove the currency symbol and convert prices to floats for math.

In [ ]:
# Remove '£' and convert to numeric
df['Price'] = df['Price'].str.replace('£', '', regex=False).astype(float)

df.head()

### Step 3: Analysis & Visualization
Now we calculate statistics and plot the data.

In [ ]:
print(f"Average Price: £{df['Price'].mean():.2f}")
print(f"Cheapest Book: £{df['Price'].min():.2f}")

plt.figure(figsize=(8, 5))
plt.hist(df['Price'], bins=10, color='teal', edgecolor='white')
plt.title('Distribution of Book Prices')
plt.xlabel('Price (£)')
plt.ylabel('Count')
plt.show()

### Step 4: Export
Save the cleaned data for future use.

In [ ]:
df.to_csv("cleaned_books.csv", index=False)
print("Data saved to cleaned_books.csv")